# Reranking Demo

Cross-encoder reranking is the highest-impact single optimization for RAG quality.

**Bi-encoder (vector search):** Query and docs embedded independently → fast but approximate  
**Cross-encoder (reranking):** Query + doc processed together → slow but accurate

**Pattern:** Retrieve top-50 with bi-encoder, rerank to top-5 with cross-encoder.

**Prerequisites:**
```bash
pip install sentence-transformers
```

No Ollama needed - uses local sentence-transformers models.

In [6]:
# Setup
import time
from sentence_transformers import SentenceTransformer, CrossEncoder

# Load models
print("Loading models...")
start = time.time()
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
print(f"  Bi-encoder loaded: {time.time()-start:.2f}s")

start = time.time()
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print(f"  Cross-encoder loaded: {time.time()-start:.2f}s")

# Sample corpus with varying relevance
DOCUMENTS = [
    {"id": "doc1", "content": "Remote work policy: Employees may work from home 3 days per week with manager approval."},
    {"id": "doc2", "content": "Office dress code: Business casual is required Monday through Thursday."},
    {"id": "doc3", "content": "WFH equipment: Company provides laptop and $100 monthly internet stipend."},
    {"id": "doc4", "content": "Vacation policy: 25 days annual leave, 5 days carry-over maximum."},
    {"id": "doc5", "content": "Remote work eligibility: Must complete 90-day probation and remote work training."},
    {"id": "doc6", "content": "Home office setup: Ergonomic chair reimbursement up to $300 available."},
    {"id": "doc7", "content": "Hybrid schedule: Core hours 10am-3pm required for remote workers."},
    {"id": "doc8", "content": "Parking policy: Employee parking is available in lot B with valid permit."},
]

print(f"\nCorpus: {len(DOCUMENTS)} documents")

Loading models...
  Bi-encoder loaded: 4.32s
  Cross-encoder loaded: 3.66s

Corpus: 8 documents


---

## 1. Bi-Encoder Retrieval (Stage 1)

In [7]:
import numpy as np

# Pre-compute document embeddings
doc_texts = [d["content"] for d in DOCUMENTS]
doc_embeddings = bi_encoder.encode(doc_texts)

def bi_encoder_search(query: str, top_k: int = 5) -> list[tuple[dict, float]]:
    """Stage 1: Fast retrieval with bi-encoder."""
    start = time.time()
    
    query_embedding = bi_encoder.encode(query)
    similarities = np.dot(doc_embeddings, query_embedding)
    
    ranked_indices = np.argsort(similarities)[::-1][:top_k]
    
    elapsed = (time.time() - start) * 1000
    
    results = [(DOCUMENTS[i], float(similarities[i])) for i in ranked_indices]
    return results, elapsed

# Test bi-encoder
query = "What are the work from home rules?"
results, latency = bi_encoder_search(query, top_k=5)

print(f"Bi-Encoder Search ({latency:.1f}ms)")
print("=" * 60)
print(f"Query: \"{query}\"\n")

for i, (doc, score) in enumerate(results):
    print(f"  {i+1}. [{score:.3f}] {doc['content'][:70]}...")

Bi-Encoder Search (28.4ms)
Query: "What are the work from home rules?"

  1. [0.505] Remote work policy: Employees may work from home 3 days per week with ...
  2. [0.335] Remote work eligibility: Must complete 90-day probation and remote wor...
  3. [0.335] Hybrid schedule: Core hours 10am-3pm required for remote workers....
  4. [0.244] Office dress code: Business casual is required Monday through Thursday...
  5. [0.238] Home office setup: Ergonomic chair reimbursement up to $300 available....


---

## 2. Cross-Encoder Reranking (Stage 2)

In [8]:
def rerank(query: str, candidates: list[tuple[dict, float]], top_k: int = 3) -> list[tuple[dict, float]]:
    """Stage 2: Accurate reranking with cross-encoder."""
    start = time.time()
    
    # Prepare pairs for cross-encoder
    pairs = [(query, doc["content"]) for doc, _ in candidates]
    
    # Score with cross-encoder
    scores = cross_encoder.predict(pairs)
    
    # Re-sort by cross-encoder scores
    reranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]
    
    elapsed = (time.time() - start) * 1000
    
    return [(doc, float(score)) for (doc, _), score in reranked], elapsed

# Apply reranking
reranked_results, rerank_latency = rerank(query, results, top_k=3)

print(f"Cross-Encoder Reranking ({rerank_latency:.1f}ms)")
print("=" * 60)
print(f"Query: \"{query}\"\n")

for i, (doc, score) in enumerate(reranked_results):
    print(f"  {i+1}. [{score:.3f}] {doc['content'][:70]}...")

Cross-Encoder Reranking (26.7ms)
Query: "What are the work from home rules?"

  1. [2.901] Remote work policy: Employees may work from home 3 days per week with ...
  2. [-7.522] Remote work eligibility: Must complete 90-day probation and remote wor...
  3. [-10.856] Office dress code: Business casual is required Monday through Thursday...


---

## 3. Before/After Comparison

In [9]:
def full_pipeline(query: str, retrieve_k: int = 5, final_k: int = 3):
    """Complete two-stage retrieval pipeline."""
    # Stage 1: Bi-encoder retrieval
    candidates, bi_latency = bi_encoder_search(query, top_k=retrieve_k)
    
    # Stage 2: Cross-encoder reranking  
    reranked, cross_latency = rerank(query, candidates, top_k=final_k)
    
    return {
        "bi_encoder_results": candidates[:final_k],
        "reranked_results": reranked,
        "bi_encoder_latency_ms": bi_latency,
        "cross_encoder_latency_ms": cross_latency,
        "total_latency_ms": bi_latency + cross_latency
    }

# Test multiple queries
test_queries = [
    "What equipment does the company provide for remote work?",
    "How many vacation days do I get?",
    "Can I work from home on Fridays?",
]

print("Pipeline Comparison: Before vs After Reranking")
print("=" * 70)

for query in test_queries:
    results = full_pipeline(query)
    
    print(f"\nQuery: \"{query}\"")
    print("-" * 60)
    
    print(f"  Before reranking (bi-encoder top-3):")
    for i, (doc, _) in enumerate(results["bi_encoder_results"]):
        print(f"    {i+1}. {doc['id']}: {doc['content'][:50]}...")
    
    print(f"  After reranking (cross-encoder top-3):")
    for i, (doc, score) in enumerate(results["reranked_results"]):
        print(f"    {i+1}. {doc['id']}: {doc['content'][:50]}... (score: {score:.2f})")
    
    print(f"  Latency: {results['bi_encoder_latency_ms']:.1f}ms + {results['cross_encoder_latency_ms']:.1f}ms = {results['total_latency_ms']:.1f}ms")

Pipeline Comparison: Before vs After Reranking

Query: "What equipment does the company provide for remote work?"
------------------------------------------------------------
  Before reranking (bi-encoder top-3):
    1. doc5: Remote work eligibility: Must complete 90-day prob...
    2. doc3: WFH equipment: Company provides laptop and $100 mo...
    3. doc1: Remote work policy: Employees may work from home 3...
  After reranking (cross-encoder top-3):
    1. doc1: Remote work policy: Employees may work from home 3... (score: -2.47)
    2. doc3: WFH equipment: Company provides laptop and $100 mo... (score: -2.82)
    3. doc7: Hybrid schedule: Core hours 10am-3pm required for ... (score: -4.07)
  Latency: 15.1ms + 14.5ms = 29.5ms

Query: "How many vacation days do I get?"
------------------------------------------------------------
  Before reranking (bi-encoder top-3):
    1. doc4: Vacation policy: 25 days annual leave, 5 days carr...
    2. doc1: Remote work policy: Employees may work 

---

## 4. Reranker Model Comparison

In [10]:
# Note: Latency varies significantly based on hardware, batch size, 
# and whether model is already loaded. The measurements above from
# this notebook are more reliable than generic benchmarks.

print("Reranker Model Options")
print("=" * 60)
print("""
Model                         Notes
----------------------------------------------------------------
cross-encoder/ms-marco-MiniLM Lightweight, good for learning (used here)
BAAI/bge-reranker-v2-m3       Higher quality, larger model
Cohere rerank-v3.5            API-only, strong benchmark performance
Jina reranker-v2              Available via API or self-hosted

Latency depends on:
- Number of candidates to rerank
- Document length
- Hardware (CPU vs GPU)
- Whether model is warm (already loaded)

See the actual measurements in this notebook for realistic expectations.
""")

Reranker Model Options

Model                         Notes
----------------------------------------------------------------
cross-encoder/ms-marco-MiniLM Lightweight, good for learning (used here)
BAAI/bge-reranker-v2-m3       Higher quality, larger model
Cohere rerank-v3.5            API-only, strong benchmark performance
Jina reranker-v2              Available via API or self-hosted

Latency depends on:
- Number of candidates to rerank
- Document length
- Hardware (CPU vs GPU)
- Whether model is warm (already loaded)

See the actual measurements in this notebook for realistic expectations.



---

## Summary

**Two-Stage Retrieval Pattern:**
1. **Stage 1 (Bi-encoder):** Retrieve top-K candidates from corpus (fast, approximate)
2. **Stage 2 (Cross-encoder):** Rerank candidates to final top-N (slower, accurate)

**Why it works:** Cross-encoders process query+document together, enabling deeper semantic matching than independent embeddings.

**Practical guidance:**
- Start with ms-marco-MiniLM for learning and prototyping
- Measure actual latency on your hardware before production
- Reranking is one of the highest-impact RAG optimizations

**Always benchmark on your own data and hardware.**